# Tutorial: Discriminative vs Generative Models

Audience:
- Students who have completed Modules 02-05 and are ready to compare supervised modeling paradigms in one compact experiment.

Prerequisites:
- Train/validation/test splits and evaluation metrics.
- Logistic regression.
- Kernel methods and SVMs.
- Gaussian Naive Bayes, latent variables, and EM.

Learning goals:
- Train two discriminative and two generative models on the same dataset.
- Compare modeling assumptions, fitting procedures, and performance.
- Write a short evidence-based analysis of when each modeling family is preferable.


## Outline

1. Load a shared binary-classification dataset.
2. Build a train/validation/test protocol.
3. Train logistic regression and an SVM.
4. Train Gaussian Naive Bayes and a class-conditional Gaussian mixture model.
5. Compare metrics, confusion matrices, and modeling assumptions.
6. Complete the written analysis prompts.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 7
np.random.seed(SEED)
SEED


## Step 1 - Load the dataset and define the split

We use the Wisconsin breast cancer dataset because it is small, tabular, and continuous-valued. That makes Gaussian assumptions visible without adding image or text preprocessing.

Before running the next cell, write a one- or two-sentence prediction:

- Which model family do you expect to perform best?
- Which model do you expect to make the strongest assumptions about the data-generating process?


In [ ]:
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target
feature_names = dataset.feature_names
target_names = dataset.target_names

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=SEED,
)

print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Train / validation / test: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"Positive-class rate: {y.mean():.3f}")
print(f"Target names: {list(target_names)}")


## Step 2 - Define an evaluation helper

We will evaluate every model on the same validation and test splits. This avoids unfair comparisons caused by different train/test partitions.


In [ ]:
@dataclass
class ModelResult:
    name: str
    model_type: str
    validation_accuracy: float
    test_accuracy: float
    precision: float
    recall: float
    f1: float
    roc_auc: float
    y_pred: np.ndarray
    y_score: np.ndarray


def evaluate_model(name, model_type, model, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        test_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        raw = model.decision_function(X_test)
        raw = np.asarray(raw, dtype=float)
        test_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    else:
        test_score = test_pred.astype(float)

    return ModelResult(
        name=name,
        model_type=model_type,
        validation_accuracy=accuracy_score(y_val, val_pred),
        test_accuracy=accuracy_score(y_test, test_pred),
        precision=precision_score(y_test, test_pred),
        recall=recall_score(y_test, test_pred),
        f1=f1_score(y_test, test_pred),
        roc_auc=roc_auc_score(y_test, test_score),
        y_pred=test_pred,
        y_score=test_score,
    )


def result_table(results):
    rows = []
    for result in results:
        rows.append(
            {
                "model": result.name,
                "type": result.model_type,
                "val_acc": round(result.validation_accuracy, 4),
                "test_acc": round(result.test_accuracy, 4),
                "precision": round(result.precision, 4),
                "recall": round(result.recall, 4),
                "f1": round(result.f1, 4),
                "roc_auc": round(result.roc_auc, 4),
            }
        )
    return rows


## Step 3 - Train two discriminative models

These models target the class label directly. Logistic regression models a conditional probability through a linear logit, while the SVM optimizes a margin-based objective. Both rely on optimization over a discriminative objective rather than a full density model.

Task:
- Record why each of these is a discriminative model.
- If you change `C` or `gamma`, justify the change briefly in markdown below the cell.


In [ ]:
discriminative_models = {
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)),
        ]
    ),
    "RBF SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=SEED)),
        ]
    ),
}

discriminative_results = []
for name, model in discriminative_models.items():
    discriminative_results.append(
        evaluate_model(name, "discriminative", model, X_train, y_train, X_val, y_val, X_test, y_test)
    )

result_table(discriminative_results)


## Step 4 - Train two generative models

Gaussian Naive Bayes assumes conditional independence of features given the class and fits one Gaussian per feature per class. The GMM classifier is more flexible: it fits a mixture density to each class and then combines class priors with class-conditional likelihoods.

Task:
- Explain why Naive Bayes and the class-conditional GMM count as generative.
- Note which assumptions are stronger or weaker than the discriminative models above.


In [ ]:
class ClassConditionalGMMClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=2, covariance_type="full", random_state=SEED):
        self.n_components = n_components
        self.covariance_type = covariance_type
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.models_ = {}
        self.log_priors_ = {}

        for cls in self.classes_:
            X_cls = X[y == cls]
            model = GaussianMixture(
                n_components=self.n_components,
                covariance_type=self.covariance_type,
                random_state=self.random_state,
            )
            model.fit(X_cls)
            self.models_[cls] = model
            self.log_priors_[cls] = np.log(len(X_cls) / len(X))
        return self

    def _joint_log_likelihood(self, X):
        scores = []
        for cls in self.classes_:
            log_density = self.models_[cls].score_samples(X)
            scores.append(log_density + self.log_priors_[cls])
        return np.column_stack(scores)

    def predict_proba(self, X):
        joint = self._joint_log_likelihood(np.asarray(X))
        joint -= joint.max(axis=1, keepdims=True)
        probs = np.exp(joint)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]


generative_models = {
    "Gaussian Naive Bayes": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", GaussianNB()),
        ]
    ),
    "Class-Conditional GMM": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", ClassConditionalGMMClassifier(n_components=2, covariance_type="full", random_state=SEED)),
        ]
    ),
}

generative_results = []
for name, model in generative_models.items():
    generative_results.append(
        evaluate_model(name, "generative", model, X_train, y_train, X_val, y_val, X_test, y_test)
    )

result_table(generative_results)


## Step 5 - Compare the results

The next cells build a compact comparison table and confusion-matrix view. Use them as evidence when you answer the written prompts.


In [ ]:
all_results = discriminative_results + generative_results
comparison_rows = result_table(all_results)
comparison_rows


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

for ax, result in zip(axes, all_results):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        result.y_pred,
        ax=ax,
        colorbar=False,
    )
    ax.set_title(result.name)

plt.tight_layout()
plt.show()


## Analysis prompts

Write your answers directly in this notebook.

1. Which model achieved the strongest held-out performance? Use at least two metrics in your answer.
2. Which model made the strongest assumptions about feature generation or independence?
3. Compare the training procedures: convex optimization, margin-based optimization, closed-form parameter estimates inside Naive Bayes, and EM for mixtures.
4. Where do the generative models seem to gain or lose compared with the discriminative ones?
5. If you needed a model for a new small-data scientific or medical problem, which family would you try first and why?


## Answer scaffold

Replace the bullets below with your final analysis.

- Best overall model:
- Most plausible reason for that result:
- Strongest generative assumption:
- Most important failure mode you observed:
- When a generative model would still be the better choice:


## Extensions

Choose one if you want to push further:

- Tune the number of GMM components and describe the tradeoff.
- Swap the RBF SVM for a linear SVM and compare decision-boundary flexibility.
- Inject synthetic missing values and discuss which models degrade most.
- Examine whether the best-performing model is also the best-calibrated model.
